# 분류기 폴백과 과금

Claude Fable 5는 사이버보안, 생물학, 화학 같은 영역에서 뛰어난 역량을 갖고 있어 오용의 위험도 실재합니다. 유용하게 만드는 바로 그 능력이 악의적인 행위자가 사이버 공격이나 위험한 무기를 만드는 데 도움이 될 수 있기 때문입니다. 그래서 Claude Fable 5에는 이 영역들에서 성능을 제한하는 안전장치가 함께 제공되고, 모든 요청에 자동 안전 검사가 실행됩니다. 이 검사는 세 영역의 요청을 차단합니다.

- **공격적 사이버보안 기법** — 익스플로잇, 악성코드, 공격 도구 제작
- **생물학과 생명과학** — 실험 방법이나 분자 메커니즘
- **모델의 [요약된 사고](https://platform.claude.com/docs/en/build-with-claude/extended-thinking#summarized-thinking) 추출**

이 안전장치는 의도적으로 보수적입니다. 견고성을 최우선으로 조정되어 있어서, 무해한 기술 작업이 걸리는 경우도 있습니다. Fable의 Mythos 수준 역량을 다른 모든 영역에서 더 빨리 제공하기 위해, 생물학과 사이버보안 관련 주제 전반에 Opus 4.8로의 폴백을 함께 두고 Fable 5를 출시합니다. 출시 이후에도 Fable 5의 거짓 양성률을 계속 낮춰 갈 것입니다.

**API 고객은 Claude Fable 5에서 Opus 4.8로의 폴백을 설정해야 합니다.** [내장 서버 측 폴백](https://platform.claude.com/docs/en/build-with-claude/handling-stop-reasons#server-side-fallback) 기능(Claude 자체 API와 AWS의 Claude Platform에서 사용 가능)을 쓰거나, Anthropic SDK 헬퍼로 만든 [클라이언트 측 폴백](https://platform.claude.com/docs/en/build-with-claude/handling-stop-reasons#client-side-fallback) 로직을 쓰면 됩니다.

또한 Fable 5 폴백이 일어나는 대부분의 경우 토큰 비용이 발생하지 않도록 과금도 변경했습니다. **서버 측 폴백 기능을 쓰지 않는 경우**에는 이 변경을 적용하기 위한 조치가 필요합니다. [아래](#4-billing-changes)를 참고하세요.

## 이 가이드가 다루는 것

1. [분류기 차단은 어떤 모습인가](#1-what-a-classifier-block-looks-like)
2. [서버 측 폴백 (권장)](#2-server-side-fallback-recommended)
3. [스트리밍](#3-streaming)
4. [과금 변경](#4-billing-changes)
5. [SDK로 하는 클라이언트 측 폴백](#5-client-side-fallback-with-the-sdk)
6. [흔한 안티패턴](#6-common-anti-patterns)

In [ ]:
%%capture
%pip install -U "anthropic>=0.108.0"

In [ ]:
import os

from dotenv import load_dotenv

load_dotenv()

PRIMARY_MODEL = "claude-fable-5"
FALLBACK_MODEL = "claude-opus-4-8"
SERVER_SIDE_FALLBACK_BETA = "server-side-fallback-2026-06-01"
FALLBACK_CREDIT_BETA = "fallback-credit-2026-06-01"

# Anthropic() reads ANTHROPIC_API_KEY from the environment. Add it to a .env
# file (loaded above) or export it in your shell before running the live examples.
if not os.environ.get("ANTHROPIC_API_KEY"):
    print(
        "ANTHROPIC_API_KEY is not set - add it to .env or export it."
    )


## 1. 분류기 차단은 어떤 모습인가

*분류기 차단*은 요청이 안전장치를 위반하는 것으로 보일 때 API가 반환하는 것입니다. API는 `200`과 함께 `stop_reason: "refusal"`, 그리고 범주를 설명하는 `stop_details` 객체를 반환합니다.

```json
{
  "stop_reason": "refusal",
  "stop_details": {
    "type": "refusal",
    "category": "cyber",
    "explanation": "This request triggered restrictions on violative cyber content and was blocked under Anthropic's Usage Policy..."
  },
  "content": [...]
}
```

로직 분기는 **`content`나 `stop_details`가 아니라 `stop_reason`**을 기준으로 하세요. `stop_details`는 참고용이며 `null`일 수 있는데, 그럴 때는 아래 범주에 해당하지 않는 일반적인 거부로 다루면 됩니다.

값이 있을 때 `category`는 `"cyber"`, `"bio"`, `"reasoning_extraction"` 중 하나입니다. 폴백 선택을 다듬는 데 활용할 수 있습니다.

| category | 발동 조건 |
| --- | --- |
| `cyber` | 공격적 사이버보안 콘텐츠(익스플로잇, 악성코드, 공격 도구) |
| `bio` | 생물학·생명과학 콘텐츠(실험 방법, 분자 메커니즘) |
| `reasoning_extraction` | 모델의 [요약된 사고](https://platform.claude.com/docs/en/build-with-claude/extended-thinking#summarized-thinking)를 추출하려는 요청 |

분류기 차단은 **모델 거부**(모델 자체가 다른 정책상의 이유로 응답을 거절하는 것)와 다릅니다. 둘 다 `stop_reason: "refusal"`로 나타나지만, `stop_details.category`가 어떤 분류기가 차단했는지 알려 줍니다.

> **참고:** 서버 측 폴백 기능을 쓰지 **않는** 경우 `stop_details`에는 `fallback_credit_token`도 포함됩니다. 폴백 모델 요청을 캐시 읽기로 과금하는 데 사용합니다. [과금 변경](#4-billing-changes)을 참고하세요.

## 2. 서버 측 폴백 (권장)

Messages API가 폴백을 대신 실행해 줄 수 있습니다. `fallbacks`에 `[{"model": "claude-opus-4-8"}]`을 넘기고 `server-side-fallback-2026-06-01` 베타 헤더를 붙이세요. Fable의 분류기가 해당 턴을 차단하면 API가 자동으로 Opus 4.8로 재시도하며, 무슨 일이 있었는지 알 수 있도록 표시가 남습니다.

자동 폴백 기능은 현재 **Claude API**와 **AWS의 Claude Platform**에서 지원됩니다. 지금은 Fable 5에서 Opus 4.8로의 폴백만 지원하며, 앞으로 확대할 예정입니다.

```bash
curl https://api.anthropic.com/v1/messages \
  -H "x-api-key: $ANTHROPIC_API_KEY" \
  -H "anthropic-version: 2023-06-01" \
  -H "anthropic-beta: server-side-fallback-2026-06-01" \
  -H "content-type: application/json" \
  -d '{
    "model": "claude-fable-5",
    "max_tokens": 1024,
    "fallbacks": [
      { "model": "claude-opus-4-8" }
    ],
    "messages": [
      { "role": "user", "content": "Hello, world" }
    ]
  }'
```

### 폴백을 실행할 수 없을 때

`fallbacks`를 설정했지만 API가 폴백 모델에 닿을 수 없을 때(요청 한도 소진, 과부하 등)에는 해당 턴이 여전히 거부로 돌아오며, `stop_details.recommended_model`이 직접 재시도할 정식 모델 ID를 알려 줍니다.

```json
{
  "stop_reason": "refusal",
  "stop_details": {
    "type": "refusal",
    "category": "cyber",
    "recommended_model": "claude-opus-4-8"
  }
}
```

`recommended_model`은 **오직** 이 경우(폴백이 설정되어 있고 *또한* 폴백을 실행할 수 없었던 경우)에만 채워집니다. 폴백을 설정하지 않은 단순 차단에서는 나타나지 않으며, 그래서 [1절](#1-what-a-classifier-block-looks-like)의 기본 예시에는 없습니다.

In [ ]:
# Fable applies extra safety filters. With a fallback chain configured, the API
# retries blocked turns on the next model server-side. A stop_reason of "refusal"
# means the whole chain refused.

from anthropic import Anthropic

client = Anthropic()


def chat_turn(messages, max_tokens=1024):
    """One API call; the server handles the fallback."""
    return client.beta.messages.create(
        model=PRIMARY_MODEL,
        max_tokens=max_tokens,
        messages=messages,
        betas=[SERVER_SIDE_FALLBACK_BETA],
        fallbacks=[{"model": FALLBACK_MODEL}],
    )


### 폴백 감지하기 (비스트리밍)

폴백 응답에는 전환 지점마다 `{"type": "fallback"}` 콘텐츠 블록이 담기고, `usage.iterations`에 모델별 사용량이 기록됩니다. 다만 **고정 처리(sticky-served)** 턴, 즉 대화의 앞선 턴이 폴백되었기 때문에 폴백 모델로 곧바로 라우팅된 턴에는 폴백 블록이 *없습니다*. 요청이 곧바로 라우팅되어 `content`에 표시할 경계가 없기 때문입니다. 폴백 모델이 해당 턴을 처리했는지 확실히 알아보는 방법은 `usage.iterations`입니다.

In [ ]:
def fallback_hops(response):
    """(from_model, to_model) for each hop that ran and blocked this turn."""
    hops = []
    for b in response.content:
        if getattr(b, "type", None) == "fallback":
            d = b.model_dump() if hasattr(b, "model_dump") else dict(b)
            hops.append((d["from"]["model"], d["to"]["model"]))
    return hops


def served_by_fallback(response):
    """True whenever a fallback model served the response, INCLUDING a
    sticky-served turn (which carries no fallback block). usage.iterations is
    the best way to check whether a turn was served by a fallback model."""
    iters = getattr(response.usage, "iterations", None) or []
    return any(
        (i.get("type") if isinstance(i, dict) else getattr(i, "type", None))
        == "fallback_message"
        for i in iters
    )


response = chat_turn([{"role": "user", "content": "Hello, world"}])
hops = fallback_hops(response)
for from_model, to_model in hops:
    print(f"[{from_model} blocked \u2014 continued on {to_model}]")
if not hops and served_by_fallback(response):
    print(f"[sticky: served directly by {response.model}]")


### 스트리밍 중 폴백 감지하기

블록이 `{"type": "fallback"}`인 `content_block_start` 이벤트를 지켜보세요. 스트림 중 전환 지점을 표시합니다. 다만 턴 단위의 확정적인 답은 비스트리밍 응답과 똑같이 **최종** 메시지의 `usage.iterations`를 확인하는 것입니다. 이 확인은 폴백 모델이 곧바로 처리한 스트림을 포함해 모든 경우에 신뢰할 수 있으므로, 스트림 중 이벤트에만 의존하지 말고 이것을 진실의 근거로 삼으세요.

In [ ]:
with client.beta.messages.stream(
    model=PRIMARY_MODEL,
    max_tokens=1024,
    messages=[{"role": "user", "content": "Hello, world"}],
    betas=[SERVER_SIDE_FALLBACK_BETA],
    fallbacks=[{"model": FALLBACK_MODEL}],
) as stream:
    for event in stream:
        if (
            getattr(event, "type", None) == "content_block_start"
            and getattr(event.content_block, "type", None) == "fallback"
        ):
            fb = event.content_block
            fb = fb.model_dump() if hasattr(fb, "model_dump") else dict(fb)
            print(f"[switching: {fb['from']['model']} -> {fb['to']['model']}]")
    final = stream.get_final_message()

# Definitive per-turn check, same as non-streaming: usage.iterations also
# catches a stream served directly by the fallback model (no in-stream event).
if served_by_fallback(final):
    print(f"[fallback model served this stream: {final.model}]")


### 폴백 응답의 형태

폴백 응답에는 `message.model`(결국 응답한 모델), 전환 지점마다 표시되는 `{"type": "fallback"}` 콘텐츠 블록, 그리고 `usage.iterations`의 시도별 사용량이 담깁니다.

```json
{
  "id": "msg_01Ab...",
  "type": "message",
  "role": "assistant",
  "model": "claude-opus-4-8",
  "content": [
    { "type": "fallback", "from": { "model": "claude-fable-5" }, "to": { "model": "claude-opus-4-8" } },
    { "type": "text", "text": "..." }
  ],
  "stop_reason": "end_turn",
  "stop_details": null,
  "usage": {
    "input_tokens": 412, "output_tokens": 264,
    "cache_read_input_tokens": 0, "cache_creation_input_tokens": 0,
    "iterations": [
      { "type": "message", "model": "claude-fable-5", "input_tokens": 408, "output_tokens": 0,
        "cache_read_input_tokens": 0, "cache_creation_input_tokens": 0 },
      { "type": "fallback_message", "model": "claude-opus-4-8", "input_tokens": 412, "output_tokens": 264,
        "cache_read_input_tokens": 0, "cache_creation_input_tokens": 0 }
    ]
  }
}
```

**시도별 재정의.** 각 폴백 항목은 해당 시도에 한해 `max_tokens`, `thinking`, `output_config`, `speed`를 재정의할 수 있습니다(`output_config`와 `speed`는 최상위 필드와 동일한 베타 헤더도 추가로 필요합니다). 항목의 재정의를 반영한 요청은 그 항목의 모델에 대한 올바른 형식의 직접 요청이어야 합니다.

```json
{
  "model": "claude-fable-5",
  "max_tokens": 1024,
  "fallbacks": [
    { "model": "claude-opus-4-8", "max_tokens": 8192, "thinking": {"type": "disabled"}, "speed": "fast" }
  ],
  "messages": [
    { "role": "user", "content": "Hello, world" }
  ]
}
```

**과금.** `usage.input_tokens`는 턴당 한 번만 계산됩니다. `usage.output_tokens`는 답변을 반영합니다. 모델별로 정확히 배분해야 한다면 `usage.iterations`를 사용하세요.

## 3. 스트리밍

스트리밍에서 폴백은 자동으로 동작하도록 설계되었습니다. **출력이 여러분에게 도달하기 전에** 분류기가 차단하면 스트림이 폴백 모델의 응답으로 시작됩니다. 이 재시도는 보이지 않으며 폴백 SSE 이벤트도 발생하지 않습니다.

**스트리밍 도중에** 분류기가 차단하면 재시도도 같은 스트림에서 일어납니다. 이미 나온 부분 출력은 그대로 유지되고, `{"type": "fallback"}` 콘텐츠 블록이 경계를 표시하며, 폴백 모델이 그 부분 출력에 이어서 계속합니다. 스트리밍된 내용은 절대 버려지지 않습니다.

In [ ]:
def stream_turn(messages, max_tokens=1024):
    with client.beta.messages.stream(
        model=PRIMARY_MODEL,
        max_tokens=max_tokens,
        messages=messages,
        betas=[SERVER_SIDE_FALLBACK_BETA],
        fallbacks=[{"model": FALLBACK_MODEL}],
    ) as stream:
        # Nothing streamed is ever discarded: after a mid-stream block, the
        # final message is partial + fallback block + continuation.
        final = stream.get_final_message()

    if final.stop_reason == "refusal":
        return final  # the whole chain refused
    text = "".join(b.text for b in final.content if b.type == "text")
    print(f"{final.model}: {text}")
    return final


## 4. 과금 변경

폴백의 비용 영향을 최소화하도록 과금을 변경했습니다. 폴백과 Anthropic SDK 헬퍼를 사용하면 자동으로 적용됩니다. **조치가 필요한 것은 캐시 미스 과금 변경뿐이며, 그것도 서버 측 폴백을 쓰지 않는 경우에만 해당합니다.**

**1. 직접적인 분류기 차단에서는 입력 토큰이 과금되지 않습니다**(즉 출력 토큰이 반환되기 전에 요청이 차단된 경우). 조치가 필요 없습니다. Fable 5를 포함한 모든 프로덕션 모델에 이미 자동으로 적용되어 있습니다.

**2. Fable 5 → Opus 4.8 폴백의 입력 토큰은 캐시 적중으로 과금됩니다.** 보통 다른 모델로 전환하면 [캐시 *쓰기*](https://platform.claude.com/docs/en/build-with-claude/prompt-caching#how-prompt-caching-works)로 과금되며, 이는 기본 입력 토큰 비용보다 1.25배(5분 TTL) 또는 2배(60분 TTL) 높습니다. 대신 이 Opus 토큰은 이미 캐싱되어 있던 것처럼, 즉 기본 입력 토큰 가격의 10%인 캐시 *읽기*로 과금합니다.

- **서버 측 폴백 기능 사용 시:** 이 과금 변경이 자동으로 적용됩니다.
- **서버 측 폴백 기능 미사용 시:** 아래의 크레딧 토큰 흐름을 참고하세요.

### 폴백 크레딧 토큰 사용하기 (클라이언트 측 폴백만 해당)

안전 분류기에 차단된 Fable 요청에는 `stop_details`에 `fallback_credit_token`이 포함됩니다. 이 토큰은 차단된 요청에 과금 대상 캐시 접두부가 있었을 때**만** 존재하며, 그렇지 않으면 `null`입니다.

사용하려면:

1. 이어지는 Opus 4.8 요청에 `anthropic-beta: fallback-credit-2026-06-01` 헤더를 붙이세요.
2. 토큰을 최상위 `fallback_credit_token` 파라미터로 전달하세요.
3. 프롬프트를 구성하는 필드를 차단된 요청과 **동일하게** 유지하세요. `system`, `messages`, `tools`가 정확히 같아야 합니다.

그러면 Fable 요청에서 캐싱되었던 접두부가 캐시 쓰기가 아니라 캐시 읽기 요율로 과금됩니다. 전환 비용이 환급되므로, 재시도 비용이 처음부터 Opus로 대화했을 때와 같아집니다.

**유효 조건:** 이 토큰은 차단된 Fable 5 요청으로부터 **5분 이내**에 발생하고 **같은 조직과 워크스페이스**에서 나온 Opus 4.8 요청에서만 유효합니다.

**스트리밍 도중 차단.** Fable 5 요청이 출력 토큰을 스트리밍하는 도중에 차단되면 `stop_details`에 크레딧 토큰과 함께 `fallback_has_prefill_claim: true`도 포함됩니다. 이 경우 이어지는 Opus 4.8 요청에서 그 부분 출력을 assistant 프리필로 덧붙여 Fable이 멈춘 지점부터 이어 갈 수 있습니다. Opus 4.8 요청에서 [평소에는 허용되지 않는](https://platform.claude.com/docs/en/test-and-evaluate/strengthen-guardrails/increase-consistency#prefill-claudes-response) 일입니다.

In [ ]:
def redeem_credit_after_block(blocked_response, messages, max_tokens=1024):
    """Retry a classifier-blocked Fable turn on Opus 4.8, redeeming the
    fallback credit token so the cached prefix is billed at the cache-read
    rate. Use this only when you are NOT using server-side fallback."""
    details = blocked_response.stop_details
    credit = getattr(details, "fallback_credit_token", None) if details else None

    extra = {}
    betas = []
    if credit is not None:  # present only when the blocked request had a cached prefix
        betas.append(FALLBACK_CREDIT_BETA)
        extra["fallback_credit_token"] = credit

    # The system, messages, and tools must be IDENTICAL to the blocked request.
    return client.beta.messages.create(
        model=FALLBACK_MODEL,
        max_tokens=max_tokens,
        messages=messages,
        betas=betas or None,
        extra_body=extra or None,
    )


## 5. SDK로 하는 클라이언트 측 폴백

서버 측 폴백은 Claude 자체 API와 AWS의 Claude Platform에서 사용할 수 있지만, 현재 Amazon Bedrock, Vertex AI, Microsoft Foundry, Message Batches API에서는 사용할 수 없습니다. 이런 환경이거나 폴백 로직을 클라이언트에 두고 싶을 때를 위해, Anthropic SDK(Python, TypeScript, Go, Java, C#)는 **거부 폴백 미들웨어**를 제공합니다.

폴백 모델 목록과 `BetaFallbackState`로 클라이언트에 한 번 설정한 뒤, 평소처럼 `client.beta.messages`를 호출하면 됩니다. 이 미들웨어는 다음을 수행합니다.

- `stop_reason: "refusal"` 턴을 목록의 다음 모델로 재시도합니다(폴백도 거부하면 체인을 따라 계속 내려가고, 모든 항목이 거부하면 예외를 던지는 대신 원래 거부를 그대로 전달합니다)
- **모든 요청에 `fallback-credit-2026-06-01` 베타 헤더를 자동으로 붙여 줍니다.** 덕분에 토큰을 직접 관리하지 않고도 [4절](#4-billing-changes)의 캐시 읽기 과금 혜택을 얻습니다
- 대화 기록의 `fallback` 콘텐츠 블록을 대신 관리해 줍니다
- 응답한 모델을 `BetaFallbackState`에 기록해 이후 턴이 그 모델에 고정되게 합니다

이 미들웨어는 서버 측 `fallbacks` 파라미터와 **함께 쓸 수 없습니다.** 둘 중 하나만 사용하세요. (미들웨어를 설치한 앱에서 서버 측 `fallbacks` 요청을 보내려면 미들웨어가 없는 별도 클라이언트 인스턴스를 쓰세요.)

In [ ]:
from anthropic import Anthropic, BetaFallbackState, BetaRefusalFallbackMiddleware

# Install the middleware once, with your fallback chain. No per-request betas needed.
client = Anthropic(
    middleware=[BetaRefusalFallbackMiddleware([{"model": FALLBACK_MODEL}])],
)

state = BetaFallbackState()  # reuse across turns to pin follow-ups to the accepting model

# Non-streaming: a refused Fable turn is retried on Opus 4.8 transparently.
with state:
    message = client.beta.messages.create(
        model=PRIMARY_MODEL,
        max_tokens=1024,
        messages=[{"role": "user", "content": "Hello, Claude"}],
    )
print(f"served by: {message.model}")

# Streaming: on a refusal the middleware splices the fallback model's events
# onto the same open stream.
with (
    state,
    client.beta.messages.stream(
        model=PRIMARY_MODEL,
        max_tokens=1024,
        messages=[{"role": "user", "content": "Hello, Claude"}],
    ) as stream,
):
    for event in stream:
        if event.type == "text":
            print(event.text, end="", flush=True)
    final = stream.get_final_message()
print(f"\nserved by: {final.model}")


## 6. 흔한 안티패턴

**계정마다 한 번이 아니라 요청마다 폴백을 설정하세요.** Opus 4.8 폴백을 켜는 계정 수준이나 세션 수준 스위치는 없습니다. 모든 API 호출에 폴백 설정이 들어가야 합니다. 폴백을 켜지 않은 호출은 조용히 폴백 모델로 재시도하는 대신 거부를 반환합니다.

**요청을 만드는 모든 코드 경로를 점검하세요.** 재시도 버튼, 메시지 재생성, 도구 사용 연속 처리 같은 기능은 각자 요청을 구성하는 경우가 많고, 각각이 조용히 폴백 설정을 빠뜨릴 수 있습니다. 모든 진입점에서 폴백을 명시적으로 설정하세요. (코드베이스 전반에 폴백을 빠르게 추가하려면 [마이그레이션 가이드](https://platform.claude.com/docs/en/about-claude/models/migration-guide)와 [claude-api 스킬](https://platform.claude.com/docs/en/agents-and-tools/agent-skills/claude-api-skill)을 참고하세요.)

**서브에이전트 호출에도 폴백을 포함하고, 에이전트별로 동작한다는 점을 예상하세요.** 한 세션에서 여러 에이전트를 돌린다면 모든 에이전트의 호출에 폴백 설정이 필요합니다. 거부가 발생하면 그것을 받은 에이전트만 폴백 모델로 옮겨 가고 나머지 에이전트는 Fable에 머무릅니다. 한 에이전트의 폴백이 세션 전체에 적용된다고 가정하지 마세요. (Claude Code의 서브에이전트도 마찬가지입니다. 거부를 만난 서브에이전트만 Opus로 폴백하고 나머지 세션은 Fable 5에서 계속됩니다.)

**거부 이후 같은 요청을 다시 보내면 또 거부됩니다.** 거부된 내용이 여전히 대화 기록에 남아 있으므로 같은 모델에 다시 보내면 차단이 또 발동합니다. `stop_reason: "refusal"`이 나오면 폴백 모델로 재시도하고, 라우터가 남은 대화 동안 폴백 모델에 머물도록 별도의 표시를 설정하세요.

**처리 모델 분석은 요청한 모델이 아니라 `usage.iterations`로 만드세요.** 응답의 `model` 필드는 실제로 응답한 모델이므로, 폴백으로 처리된 턴은 Opus 4.8로 보고됩니다. *요청한* 모델을 기준으로 기록한 분석은 폴백이 일어날 때마다 틀리게 됩니다. 턴 단위로 신뢰할 수 있는 확인 방법은 최종 usage 레코드의 `usage.iterations`입니다.

**스트리밍 중단을 신중히 처리하세요.** 스트리밍 도중 분류기 차단을 만나면 폴백 이전에 나온 `thinking`, `tool_use` 등의 블록을 빼세요. 잘린 `tool_use` 블록은 파싱할 수 없는 JSON이고, 다른 모델의 `thinking` 블록은 다음 호출을 깨뜨립니다. 서버 측 폴백 없이 이어 간다면 부분 응답을 완성된 assistant 턴에 붙여 넣는 대신 `stop_details`의 `fallback_has_prefill_claim` 권한을 사용하세요.